# Lab 1: NetworkX

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bonsai-lab-polimi/PyNet2Plan-lab/blob/main/labs/lab1_exercises.ipynb)

*Communication Network Design, laboratory. This notebook accompanies handout 2, "Networks as NetworkX graphs", and contains the same exercises as the script `labs/lab1_exercises.py`. Work in the form you prefer.*

## What this notebook is about

In the lab a network is a **graph**: the cities are its **nodes** and the optical links between them are its **edges**. PyNet2Plan, the tool of the lab, stores every network as a graph of the Python library [NetworkX](https://networkx.org), and every planning algorithm, such as the ILP formulations of handout 3, reads the network through NetworkX calls. Before writing formulations it is therefore worth learning how to ask a graph simple questions: which links leave a node, which path is the shortest, what happens when a fibre is cut, how much traffic must cross a group of links.

The notebook has two parts.

| part | content | what you do |
|---|---|---|
| **Part 1** (sections 1 to 8) | a guided introduction to NetworkX on the network of the lab, already solved | run the cells in order and read the explanations and the outputs. There is nothing to write. |
| **Part 2** (exercises 1 to 7) | seven questions, each answered by completing a Python function | write the code, then run it. Exercises 1 to 5 are done in the lab, 6 and 7 are optional. |

The network is **Spain-7** (7 cities, 40 wavelengths on every link) and the traffic is the matrix of the lab (24 demands, 229 wavelengths in total), the same data as in handouts 1 and 3.

## How to work with the notebook

1. **Open it** in Google Colab (link above, nothing to install), or in Jupyter or VS Code on your computer. The notebook needs only the libraries `networkx` and `matplotlib`, which Colab already has and which are installed in the virtual environment of the lab.
2. **Run the three setup cells first** (data, toolbox, tools). To run a cell, click on it and press **Shift+Enter**: the cell runs and the next one is selected.
3. **Part 1.** Run the cells **in order**, from top to bottom. Every code cell is preceded by an explanation of what it does and of what to look at in its output.
4. **Part 2.** Run the cell of the helpers, then work on one exercise at a time:
    - read the text of the exercise,
    - write your code inside the function, where the comments indicate, and **delete the line** `raise NotCompletedError(...)`,
    - run the cell of the function: this only *defines* the function, and nothing is printed,
    - run the cell `run(exercise_n)` below it, which *executes* the function and prints its results.

**If the session restarts** (Colab disconnects after some time of inactivity) all the variables are lost. Use *Runtime > Run before* in Colab, or run the setup cells again, before continuing.

**Saving your work.** Colab does not save your changes in the original notebook: use *File > Save a copy in Drive*, or *File > Download > Download .ipynb*. In Jupyter or VS Code save the file as usual.

## Words used in this notebook

| word | meaning | in Python |
|---|---|---|
| node | a city of the network | a string, for example `"Madrid"` |
| link | a fibre in **one** direction, from an origin node to a destination node | a pair `(u, v)`, for example `("Madrid", "Sevilla")` |
| fibre pair | the two links between two cities, one per direction. A **fibre cut** interrupts both | `(u, v)` and `(v, u)` |
| λ | a wavelength, that is one optical channel of a fibre | |
| capacity | the wavelengths of a link: number of fibres × wavelengths per fibre. In Spain-7 every link has 1 fibre of 40 λ | `G.edges[u, v]["fibers"] * G.edges[u, v]["capacity"]` |
| demand | traffic to carry from an origin node to a destination node, in wavelengths | `d.origin`, `d.destination`, `d.traffic` |
| path | a sequence of nodes, each linked to the next one | a list, for example `["Sevilla", "Madrid", "Valencia"]` |
| hops | the number of links of a path, that is its number of nodes minus 1 | `len(path) - 1` |

The attribute with the number of fibres is called `fibers`, with the American spelling, as in PyNet2Plan.

## Setup: run these three cells first

The three cells below prepare everything that the rest of the notebook uses. You do not need to understand their code to do the exercises, but you must **run them, in order, before anything else**.

| cell | what it contains | what it creates |
|---|---|---|
| 1. Data | the networks and the traffic matrices of the folder `data/` of the lab code, written as Python dictionaries | `TOPOLOGIES`, `TRAFFIC_FILES`, `CITIES` |
| 2. Toolbox | small versions of the functions of PyNet2Plan used in the exercises (`network.*` and `traffic.*`) | the functions, the network `G` and the traffic `D` |
| 3. Tools | two functions of this notebook | `run(exercise)` and `draw(H, paths)` |

In Colab the code of the first two cells is hidden and only their title is shown. Click *Show code*, or double click the cell, to see it.

The first cell contains four networks (Spain-7, NSFNET-14, COST239 and GEANT2), two traffic matrices of Spain-7 (the traffic of the lab, `spain7_traffic_split_vs_unsplit.csv`, and a second one, `spain7_traffic.csv`) and the coordinates of four extra cities (Bilbao, Lisbon, Porto, Paris). It prints nothing. After the second cell you should read

```text
Spain-7: 7 nodes, 16 links (16 up), capacity 40 λ
24 demands, 229 wavelengths
```

If you get instead `NameError: name 'TOPOLOGIES' is not defined`, the first cell has not been run.

In [ ]:
# @title 1. Data of the lab: networks and traffic matrices of the folder data (run this cell)
# This cell only defines data and prints nothing. It contains three dictionaries.
#
#   TOPOLOGIES     file name -> network. For each network: its name, the capacity of every link
#                  (wavelengths per fibre), the nodes as (name, latitude, longitude) and the links
#                  as (origin, destination, length in km), with a fourth value, the number of
#                  fibres, when a link has more than one. The two directions of a fibre pair are
#                  two separate links.
#   TRAFFIC_FILES  file name -> traffic matrix, as a list of demands
#                  (id, origin, destination, wavelengths).
#   CITIES         name -> (latitude, longitude) of a few extra cities, used by network.add_city.
#
# The file names, such as "data/topologies/spain7.json", are those of the folder data/ of the lab
# code, so the exercises use the same names here and in the script labs/lab1_exercises.py.
# You do not need to read the values below.
TOPOLOGIES = {
    "data/topologies/spain7.json": {
        "name": "Spain-7", "capacity": 40,
        "nodes": [
            ("Madrid", 40.4168, -3.7038),
            ("Barcelona", 41.3874, 2.1686),
            ("Valencia", 39.4699, -0.3763),
            ("Sevilla", 37.3891, -5.9845),
            ("Zaragoza", 41.6488, -0.8891),
            ("Malaga", 36.7213, -4.4214),
            ("Murcia", 37.9922, -1.1307),
        ],
        "links": [
            ("Madrid", "Zaragoza", 272.94), ("Madrid", "Sevilla", 390.21), ("Madrid", "Valencia", 302.56),
            ("Barcelona", "Zaragoza", 256.21), ("Barcelona", "Valencia", 303.06), ("Valencia", "Barcelona", 303.06),
            ("Valencia", "Murcia", 176.86), ("Valencia", "Madrid", 302.56), ("Sevilla", "Malaga", 157.33),
            ("Sevilla", "Madrid", 390.21), ("Zaragoza", "Madrid", 272.94), ("Zaragoza", "Barcelona", 256.21),
            ("Malaga", "Murcia", 323.34), ("Malaga", "Sevilla", 157.33), ("Murcia", "Valencia", 176.86),
            ("Murcia", "Malaga", 323.34),
        ],
    },
    "data/topologies/nsfnet14.json": {
        "name": "NSFNET-14", "capacity": 40,
        "nodes": [
            ("Seattle", 47.6062, -122.3321),
            ("Palo Alto", 37.4419, -122.143),
            ("San Diego", 32.7157, -117.1611),
            ("Salt Lake City", 40.7608, -111.891),
            ("Boulder", 40.015, -105.2705),
            ("Houston", 29.7604, -95.3698),
            ("Lincoln", 40.8136, -96.7026),
            ("Champaign", 40.1164, -88.2434),
            ("Pittsburgh", 40.4406, -79.9959),
            ("Atlanta", 33.749, -84.388),
            ("Ann Arbor", 42.2808, -83.743),
            ("Ithaca", 42.444, -76.5019),
            ("Princeton", 40.3573, -74.6672),
            ("College Park", 38.9807, -76.9369),
        ],
        "links": [
            ("Seattle", "Palo Alto", 1130.33), ("Seattle", "San Diego", 1712.05), ("Seattle", "Champaign", 2831.22),
            ("Palo Alto", "Seattle", 1130.33), ("Palo Alto", "San Diego", 693.81), ("Palo Alto", "Salt Lake City", 957.77),
            ("San Diego", "Seattle", 1712.05), ("San Diego", "Palo Alto", 693.81), ("San Diego", "Houston", 2093.86),
            ("Salt Lake City", "Palo Alto", 957.77), ("Salt Lake City", "Boulder", 566.68), ("Salt Lake City", "Ann Arbor", 2338.82),
            ("Boulder", "Salt Lake City", 566.68), ("Boulder", "Houston", 1452.47), ("Boulder", "Lincoln", 730.48),
            ("Houston", "San Diego", 2093.86), ("Houston", "Boulder", 1452.47), ("Houston", "Atlanta", 1128.27),
            ("Houston", "College Park", 1972.43), ("Lincoln", "Boulder", 730.48), ("Lincoln", "Champaign", 719.53),
            ("Champaign", "Seattle", 2831.22), ("Champaign", "Lincoln", 719.53), ("Champaign", "Pittsburgh", 700.32),
            ("Pittsburgh", "Champaign", 700.32), ("Pittsburgh", "Atlanta", 839.58), ("Pittsburgh", "Ithaca", 366.61),
            ("Pittsburgh", "Princeton", 451.26), ("Atlanta", "Houston", 1128.27), ("Atlanta", "Pittsburgh", 839.58),
            ("Ann Arbor", "Salt Lake City", 2338.82), ("Ann Arbor", "Ithaca", 595.04), ("Ann Arbor", "Princeton", 787.1),
            ("Ithaca", "Pittsburgh", 366.61), ("Ithaca", "Ann Arbor", 595.04), ("Ithaca", "College Park", 386.84),
            ("Princeton", "Pittsburgh", 451.26), ("Princeton", "Ann Arbor", 787.1), ("Princeton", "College Park", 247.31),
            ("College Park", "Houston", 1972.43), ("College Park", "Ithaca", 386.84), ("College Park", "Princeton", 247.31),
        ],
    },
    "data/topologies/cost239.json": {
        "name": "COST239", "capacity": 40,
        "nodes": [
            ("Amsterdam", 52.3676, 4.9041),
            ("Berlin", 52.52, 13.405),
            ("Brussels", 50.8503, 4.3517),
            ("Copenhagen", 55.6761, 12.5683),
            ("London", 51.5074, -0.1278),
            ("Luxembourg", 49.6116, 6.1319),
            ("Milan", 45.4642, 9.19),
            ("Paris", 48.8566, 2.3522),
            ("Prague", 50.0755, 14.4378),
            ("Vienna", 48.2082, 16.3738),
            ("Zurich", 47.3769, 8.5417),
        ],
        "links": [
            ("Amsterdam", "London", 357.89), ("Amsterdam", "Berlin", 576.09), ("Amsterdam", "Brussels", 172.97),
            ("Amsterdam", "Luxembourg", 318.26), ("Amsterdam", "Copenhagen", 620.72), ("Berlin", "Amsterdam", 576.09),
            ("Berlin", "Copenhagen", 355.15), ("Berlin", "Prague", 281.13), ("Berlin", "Vienna", 523.55),
            ("Berlin", "Paris", 877.46), ("Brussels", "London", 320.63), ("Brussels", "Amsterdam", 172.97),
            ("Brussels", "Paris", 263.98), ("Brussels", "Luxembourg", 187.09), ("Brussels", "Milan", 697.85),
            ("Copenhagen", "London", 955.56), ("Copenhagen", "Amsterdam", 620.72), ("Copenhagen", "Berlin", 355.15),
            ("Copenhagen", "Prague", 635.21), ("London", "Amsterdam", 357.89), ("London", "Brussels", 320.63),
            ("London", "Paris", 343.56), ("London", "Copenhagen", 955.56), ("Luxembourg", "Amsterdam", 318.26),
            ("Luxembourg", "Brussels", 187.09), ("Luxembourg", "Paris", 286.95), ("Luxembourg", "Prague", 597.51),
            ("Luxembourg", "Zurich", 305.38), ("Milan", "Brussels", 697.85), ("Milan", "Paris", 639.57),
            ("Milan", "Zurich", 218.41), ("Milan", "Vienna", 625.5), ("Paris", "London", 343.56),
            ("Paris", "Berlin", 877.46), ("Paris", "Brussels", 263.98), ("Paris", "Luxembourg", 286.95),
            ("Paris", "Zurich", 487.88), ("Paris", "Milan", 639.57), ("Prague", "Berlin", 281.13),
            ("Prague", "Copenhagen", 635.21), ("Prague", "Luxembourg", 597.51), ("Prague", "Vienna", 250.87),
            ("Prague", "Zurich", 526.14), ("Vienna", "Berlin", 523.55), ("Vienna", "Milan", 625.5),
            ("Vienna", "Prague", 250.87), ("Vienna", "Zurich", 592.07), ("Zurich", "Luxembourg", 305.38),
            ("Zurich", "Paris", 487.88), ("Zurich", "Milan", 218.41), ("Zurich", "Prague", 526.14),
            ("Zurich", "Vienna", 592.07),
        ],
    },
    "data/topologies/real/top22_geant2.json": {
        "name": "GEANT2 (real topology #22)", "capacity": 40,
        "nodes": [
            ("Lisbon", 38.7077507, -9.1365919),
            ("Madrid", 40.4167047, -3.7035825),
            ("Paris", 48.8588897, 2.320041021720077),
            ("London", 51.5074456, -0.1277653),
            ("Dublin", 53.3493795, -6.2605593),
            ("Brussels", 50.8465573, 4.351697),
            ("Amsterdam", 52.3730796, 4.8924534),
            ("Luxembourg", 49.6112768, 6.129799),
            ("Bern", 46.9484742, 7.4521749),
            ("Rome", 41.8933203, 12.4829321),
            ("Valetta", 35.8989818, 14.5136759),
            ("Athens", 37.98, 23.72),
            ("Nicosia", 35.1746503, 33.3638783),
            ("Ankara", 39.9207759, 32.8540497),
            ("Sofia", 42.96, 23.25),
            ("Bucharest", 44.4361414, 26.1027202),
            ("Budapest", 47.4978789, 19.0402383),
            ("Zagreb", 45.8130967, 15.9772795),
            ("Vienna", 48.2083537, 16.3725042),
            ("Bratislava", 48.1516988, 17.1093063),
            ("Ljubljana", 46.0500268, 14.5069289),
            ("Prague", 50.0874654, 14.4212535),
            ("Berlin", 52.5170365, 13.3888599),
            ("Copenhagen", 55.6867243, 12.5700724),
            ("Oslo", 59.9133301, 10.7389701),
            ("Stockholm", 59.3251172, 18.0710935),
            ("Helsinki", 60.1674881, 24.9427473),
            ("Tallinn", 59.4372155, 24.7453688),
            ("Vilnius", 54.6870458, 25.2829111),
            ("Moscow", 55.7505412, 37.6174782),
            ("Riga", 56.9493977, 24.1051846),
            ("Warsaw", 52.2337172, 21.07143223563649),
            ("Manchester", 53.4794892, -2.2451148),
            ("Cardiff", 51.4816546, -3.1791934),
            ("Mariehamn", 60.102423, 19.94126),
        ],
        "links": [
            ("Lisbon", "Madrid", 754.35), ("Lisbon", "London", 1982.99), ("Madrid", "Lisbon", 754.35),
            ("Madrid", "Paris", 1500.0), ("Madrid", "Rome", 1703.76), ("Madrid", "Bern", 1500.0),
            ("Paris", "London", 513.25), ("Paris", "Madrid", 1500.0), ("Paris", "Bern", 656.17),
            ("Paris", "Luxembourg", 433.43, 2), ("London", "Lisbon", 1982.99), ("London", "Manchester", 392.93),
            ("London", "Cardiff", 316.88), ("London", "Brussels", 481.11), ("London", "Paris", 513.25),
            ("Dublin", "Manchester", 399.72), ("Dublin", "Cardiff", 441.87), ("Brussels", "London", 481.11),
            ("Brussels", "Amsterdam", 260.7), ("Amsterdam", "Brussels", 260.7), ("Amsterdam", "Copenhagen", 932.52),
            ("Amsterdam", "Berlin", 863.61), ("Amsterdam", "Valetta", 2477.68), ("Luxembourg", "Paris", 433.43, 2),
            ("Luxembourg", "Berlin", 901.5), ("Bern", "Paris", 656.17), ("Bern", "Madrid", 1500.0),
            ("Bern", "Berlin", 1127.12), ("Bern", "Rome", 1033.95), ("Rome", "Madrid", 1703.76),
            ("Rome", "Bern", 1033.95), ("Rome", "Vienna", 1148.31), ("Rome", "Athens", 1576.5),
            ("Rome", "Nicosia", 2446.82), ("Rome", "Valetta", 1033.89), ("Valetta", "Amsterdam", 2477.68),
            ("Valetta", "Rome", 1033.89), ("Athens", "Rome", 1576.5), ("Athens", "Nicosia", 1372.5),
            ("Athens", "Sofia", 828.0), ("Athens", "Vienna", 1603.0), ("Nicosia", "Rome", 2446.82),
            ("Nicosia", "Athens", 1372.5), ("Ankara", "Bucharest", 1123.37), ("Ankara", "Sofia", 1302.0),
            ("Sofia", "Athens", 828.0), ("Sofia", "Ankara", 1302.0), ("Sofia", "Bucharest", 423.0),
            ("Sofia", "Budapest", 903.75), ("Bucharest", "Ankara", 1123.37), ("Bucharest", "Sofia", 423.0),
            ("Bucharest", "Budapest", 964.39), ("Budapest", "Sofia", 903.75), ("Budapest", "Bucharest", 964.39),
            ("Budapest", "Bratislava", 242.16), ("Budapest", "Zagreb", 449.31), ("Zagreb", "Budapest", 449.31),
            ("Zagreb", "Ljubljana", 175.08), ("Vienna", "Rome", 1148.31), ("Vienna", "Athens", 1603.0),
            ("Vienna", "Ljubljana", 417.6), ("Vienna", "Berlin", 785.47), ("Vienna", "Bratislava", 82.48),
            ("Bratislava", "Budapest", 242.16), ("Bratislava", "Vienna", 82.48), ("Bratislava", "Prague", 436.23),
            ("Ljubljana", "Zagreb", 175.08), ("Ljubljana", "Vienna", 417.6), ("Prague", "Bratislava", 436.23),
            ("Prague", "Warsaw", 781.92), ("Prague", "Berlin", 419.28), ("Berlin", "Amsterdam", 863.61),
            ("Berlin", "Luxembourg", 901.5), ("Berlin", "Bern", 1127.12), ("Berlin", "Vienna", 785.47),
            ("Berlin", "Prague", 419.28), ("Berlin", "Warsaw", 783.33), ("Berlin", "Copenhagen", 534.71),
            ("Berlin", "Moscow", 2012.44), ("Copenhagen", "Amsterdam", 932.52), ("Copenhagen", "Berlin", 534.71),
            ("Copenhagen", "Tallinn", 1253.43), ("Copenhagen", "Moscow", 1950.25), ("Copenhagen", "Stockholm", 781.29),
            ("Copenhagen", "Oslo", 723.44), ("Oslo", "Copenhagen", 723.44), ("Oslo", "Stockholm", 625.89),
            ("Stockholm", "Copenhagen", 781.29), ("Stockholm", "Oslo", 625.89), ("Stockholm", "Mariehamn", 203.84),
            ("Stockholm", "Helsinki", 594.0), ("Helsinki", "Mariehamn", 415.44), ("Helsinki", "Stockholm", 594.0),
            ("Tallinn", "Copenhagen", 1253.43), ("Tallinn", "Riga", 418.74), ("Vilnius", "Riga", 393.13),
            ("Vilnius", "Warsaw", 584.91), ("Moscow", "Berlin", 2012.44), ("Moscow", "Copenhagen", 1950.25),
            ("Riga", "Tallinn", 418.74), ("Riga", "Vilnius", 393.13), ("Warsaw", "Prague", 781.92),
            ("Warsaw", "Berlin", 783.33), ("Warsaw", "Vilnius", 584.91), ("Manchester", "London", 392.93),
            ("Manchester", "Dublin", 399.72), ("Cardiff", "London", 316.88), ("Cardiff", "Dublin", 441.87),
            ("Mariehamn", "Stockholm", 203.84), ("Mariehamn", "Helsinki", 415.44),
        ],
    },
}

TRAFFIC_FILES = {
    "data/traffic/spain7_traffic_split_vs_unsplit.csv": [
        (0, "Sevilla", "Barcelona", 8), (1, "Barcelona", "Madrid", 15), (2, "Madrid", "Sevilla", 10),
        (3, "Murcia", "Valencia", 5), (4, "Zaragoza", "Barcelona", 12), (5, "Madrid", "Valencia", 8),
        (6, "Malaga", "Barcelona", 9), (7, "Malaga", "Valencia", 6), (8, "Valencia", "Malaga", 5),
        (9, "Madrid", "Malaga", 16), (10, "Zaragoza", "Malaga", 14), (11, "Sevilla", "Malaga", 13),
        (12, "Valencia", "Zaragoza", 7), (13, "Barcelona", "Valencia", 13), (14, "Barcelona", "Zaragoza", 5),
        (15, "Barcelona", "Murcia", 6), (16, "Sevilla", "Murcia", 14), (17, "Sevilla", "Zaragoza", 15),
        (18, "Murcia", "Sevilla", 4), (19, "Zaragoza", "Sevilla", 10), (20, "Sevilla", "Valencia", 9),
        (21, "Murcia", "Zaragoza", 5), (22, "Malaga", "Sevilla", 5), (23, "Valencia", "Madrid", 15),
    ],
    "data/traffic/spain7_traffic.csv": [
        (0, "Madrid", "Barcelona", 6), (1, "Madrid", "Valencia", 6), (2, "Madrid", "Sevilla", 6),
        (3, "Madrid", "Zaragoza", 1), (4, "Madrid", "Malaga", 4), (5, "Madrid", "Murcia", 3),
        (6, "Barcelona", "Madrid", 4), (7, "Barcelona", "Valencia", 1), (8, "Barcelona", "Sevilla", 2),
        (9, "Barcelona", "Zaragoza", 3), (10, "Barcelona", "Malaga", 2), (11, "Barcelona", "Murcia", 3),
        (12, "Valencia", "Madrid", 1), (13, "Valencia", "Barcelona", 1), (14, "Valencia", "Sevilla", 5),
        (15, "Valencia", "Zaragoza", 3), (16, "Valencia", "Malaga", 4), (17, "Valencia", "Murcia", 5),
        (18, "Sevilla", "Madrid", 3), (19, "Sevilla", "Barcelona", 1), (20, "Sevilla", "Valencia", 5),
        (21, "Sevilla", "Zaragoza", 4), (22, "Sevilla", "Malaga", 4), (23, "Sevilla", "Murcia", 4),
        (24, "Zaragoza", "Madrid", 6), (25, "Zaragoza", "Barcelona", 3), (26, "Zaragoza", "Valencia", 5),
        (27, "Zaragoza", "Sevilla", 3), (28, "Zaragoza", "Malaga", 4), (29, "Zaragoza", "Murcia", 4),
        (30, "Malaga", "Madrid", 2), (31, "Malaga", "Barcelona", 1), (32, "Malaga", "Valencia", 3),
        (33, "Malaga", "Sevilla", 5), (34, "Malaga", "Zaragoza", 4), (35, "Malaga", "Murcia", 3),
        (36, "Murcia", "Madrid", 4), (37, "Murcia", "Barcelona", 2), (38, "Murcia", "Valencia", 2),
        (39, "Murcia", "Sevilla", 2), (40, "Murcia", "Zaragoza", 5), (41, "Murcia", "Malaga", 2),
    ],
}

CITIES = {
    "Bilbao": (43.263, -2.935),
    "Lisbon": (38.7223, -9.1393),
    "Porto": (41.1579, -8.6291),
    "Paris": (48.8566, 2.3522),
}

### Setup, cell 2: the toolbox

PyNet2Plan is not installed in Colab, so this cell re-creates, in a few lines, the functions of PyNet2Plan that the exercises use. They have the same names and give the same results as the real ones, used by the script `labs/lab1_exercises.py`. They are grouped as in PyNet2Plan, so that you can write `network.load_topology(...)` or `traffic.random_traffic(...)` in both places.

| function | what it does |
|---|---|
| `network.load_topology(path)` | returns the network of a topology file as a `networkx.DiGraph` |
| `network.describe(G)` | one line describing a network: nodes, links and capacity |
| `network.add_city(G, name, allow_online=False)` | adds a node at the coordinates of a known city (dictionary `CITIES`) |
| `network.add_link(G, u, v, capacity=40, bidirectional=True)` | adds the link u → v and, with `bidirectional=True`, also v → u. The length is computed from the coordinates |
| `network.save_topology(G, path)` | saves a network in a file that PyNet2Plan opens with *File > Open topology...* |
| `network.from_networkx(H, capacity=40)` | turns a graph made by a generator of NetworkX into a PyNet2Plan network |
| `traffic.load_traffic(path, G)` | returns the demands of a traffic file, as a list |
| `traffic.total_traffic(D)` | the total number of wavelengths of a list of demands |
| `traffic.random_traffic(G, n, min, max, seed=s)` | n random demands of min to max wavelengths, always the same for the same seed |
| `traffic.save_traffic(D, path)` | saves demands in a file that PyNet2Plan opens with *Traffic > Open traffic matrix...* |

At the end the cell loads the two objects used everywhere in the notebook:

- **`G`**, the network Spain-7, a `networkx.DiGraph` with 7 nodes and 16 links,
- **`D`**, the traffic matrix of the lab, a list of 24 demands. Every demand `d` has four fields, `d.id`, `d.origin`, `d.destination` and `d.traffic` (in wavelengths).

The cell also imports the modules used in the rest of the notebook: `nx` (NetworkX), `plt` (Matplotlib), `math`, `itertools` and `os`.

In [ ]:
# @title 2. Toolbox: the functions of PyNet2Plan used by the exercises (run this cell)
# The functions below behave as network.* and traffic.* of PyNet2Plan. They read the data of the
# cell above instead of files, and they write files that the graphical interface of PyNet2Plan
# opens. You can use them without reading their code: the table above the cell says what each
# one does.

# ---- Python modules used in the notebook --------------------------------------------------------
import csv                       # writes CSV files (traffic matrices)
import itertools                 # combinations of nodes (cut bounds)
import json                      # writes JSON files (topologies)
import math                      # ceil (rounding up), trigonometry for the distances
import os                        # file names and folders
import random                    # random numbers from a seed (random traffic)
import types                     # SimpleNamespace: the functions as network.* and traffic.*
from collections import namedtuple

import matplotlib.pyplot as plt  # figures
import networkx as nx            # graphs: every NetworkX function is written nx.<name>(...)

# ---- the files used in the notebook -------------------------------------------------------------
ROOT = "."                                       # paths such as ROOT/data/... refer to the data above
# the network and the traffic of the lab
TOPOLOGY = os.path.join(ROOT, "data", "topologies", "spain7.json")
TRAFFIC = os.path.join(ROOT, "data", "traffic", "spain7_traffic_split_vs_unsplit.csv")
OUT = os.path.join("results", "lab1_exercises")  # files written by the exercises (Files panel in Colab)


class NotCompletedError(Exception):
    """The error of the exercises that are still to be completed (see the tool run, below)."""


class Demand(namedtuple("Demand", "id origin destination traffic")):
    """A demand: ``traffic`` wavelengths from ``origin`` to ``destination``.

    It is a named tuple, whose fields are read as d.id, d.origin, d.destination and d.traffic."""

    def __str__(self):                           # the text shown by print(d)
        return f"d{self.id}: {self.origin}->{self.destination} ({self.traffic} λ)"


def _embedded(path, files):
    """The data of the file ``path`` in the dictionary ``files`` of the cell above."""
    key = os.path.normpath(path).replace(os.sep, "/")        # "./data/x.json" -> "data/x.json"
    if key not in files:
        raise FileNotFoundError(f"{path}: not among the files of this notebook ({', '.join(files)})")
    return files[key]


def load_topology(path):
    """The network of a topology file (embedded above), as a networkx.DiGraph.

    Every node has the attributes lat and lon (coordinates) and up (True: working). Every link
    has the attributes capacity (wavelengths per fibre), fibers (number of fibres), length (km)
    and up."""
    data = _embedded(path, TOPOLOGIES)
    G = nx.DiGraph(name=data["name"])                        # an empty directed graph, with a name
    for name, lat, lon in data["nodes"]:
        G.add_node(name, lat=lat, lon=lon, up=True)          # one node per city
    for u, v, length, *fibers in data["links"]:              # fibers: the optional fourth value
        G.add_edge(u, v, capacity=data["capacity"], fibers=fibers[0] if fibers else 1, length=length, up=True)
    return G


def load_traffic(path, G=None):
    """The demands of a traffic file (embedded above), as a list of Demand.

    If the network G is given, the origin and the destination of every demand must be nodes of G."""
    demands = [Demand(*row) for row in _embedded(path, TRAFFIC_FILES)]
    for d in demands:
        if G is not None and (d.origin not in G or d.destination not in G):
            raise ValueError(f"demand {d}: unknown node")
    return demands


def total_traffic(demands):
    """The total number of wavelengths of a list of demands."""
    return sum(d.traffic for d in demands)


def describe(G):
    """One line describing the network, as in PyNet2Plan: name, nodes, links and capacity."""
    links = [a for _, _, a in G.edges(data=True)]            # the attributes of every link
    caps = [a.get("capacity", 0) for a in links] or [0]
    fib = [a.get("fibers", 1) for a in links] or [0]
    up = sum(1 for a in links if a.get("up", True))          # the links that are not failed
    cap = f"{min(caps)}" if min(caps) == max(caps) else f"{min(caps)}-{max(caps)}"
    fibers = "" if max(fib) <= 1 else (f", {min(fib)}" + (f"-{max(fib)}" if min(fib) != max(fib) else "")
                                       + " fibers per link")
    return (f"{G.graph.get('name', '')}: {G.number_of_nodes()} nodes, {G.number_of_edges()} links ({up} up), "
            f"capacity {cap} λ{' per fiber' if fibers else ''}{fibers}")


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance, in kilometres, between two points given by latitude and longitude."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2 - lat1) / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return 2 * 6371.0088 * math.asin(min(1.0, math.sqrt(a)))   # radius of the Earth: 6371 km


def add_city(G, city, name=None, allow_online=False):
    """Adds a node at the coordinates of ``city``, one of the cities of CITIES, and returns
    its name."""
    if city not in CITIES:
        raise KeyError(f"{city}: unknown city. Known: {', '.join(CITIES)} (or use G.add_node(name, lat=..., lon=...))")
    lat, lon = CITIES[city]
    name = name or city                      # by default, the node has the name of the city
    G.add_node(name, lat=lat, lon=lon, up=True)
    return name


def link_length(G, u, v):
    """Kilometres between two nodes with lat and lon, otherwise the distance between their
    positions x and y."""
    a, b = G.nodes[u], G.nodes[v]
    if a.get("lat") is not None and b.get("lat") is not None:
        return round(haversine_km(a["lat"], a["lon"], b["lat"], b["lon"]), 2) or 1.0
    # a network without coordinates: the distance between the positions x and y
    return round(math.hypot(a["x"] - b["x"], a["y"] - b["y"]), 3) or 1.0


def add_link(G, origin, destination, capacity=40, length=None, bidirectional=False, fibers=1):
    """Adds the link origin -> destination, and also destination -> origin with bidirectional=True.

    Without ``length``, the length is computed from the positions of the two nodes."""
    for u, v in [(origin, destination)] + ([(destination, origin)] if bidirectional else []):
        if G.has_edge(u, v):
            raise ValueError(f"the link {u}->{v} already exists")
        G.add_edge(u, v, capacity=capacity, fibers=fibers,
                   length=link_length(G, u, v) if length is None else float(length), up=True)


def random_traffic(G, num_demands, min_traffic=1, max_traffic=8, seed=0):
    """num_demands demands between distinct random pairs of nodes, each of a random number of
    wavelengths between min_traffic and max_traffic. The same seed always gives the same demands."""
    rnd = random.Random(seed)                         # a random generator, started from the seed
    pairs = [(a, b) for a in G.nodes for b in G.nodes if a != b]     # every ordered pair of nodes
    demands = []
    for i in range(min(num_demands, len(pairs))):            # partial Fisher-Yates shuffle:
        j = i + int(rnd.random() * (len(pairs) - i))         # draw one of the pairs not used yet
        pairs[i], pairs[j] = pairs[j], pairs[i]
        # a whole number of wavelengths, from min_traffic to max_traffic
        traffic = min_traffic + int(rnd.random() * (max_traffic - min_traffic + 1))
        demands.append(Demand(i, pairs[i][0], pairs[i][1], traffic))
    return demands


def save_topology(G, path):
    """Saves the network as a PyNet2Plan topology file (File > Open topology... in the tool)."""
    nodes = []
    for n, a in G.nodes(data=True):                          # every node with its attributes
        item = {"name": n}
        item.update({k: a[k] for k in ("lat", "lon", "x", "y") if a.get(k) is not None})
        item["up"] = bool(a.get("up", True))
        nodes.append(item)
    links = []
    for u, v, a in G.edges(data=True):                       # every link with its attributes
        item = {"origin": u, "destination": v, "capacity": int(a.get("capacity", 40))}
        if int(a.get("fibers", 1)) != 1:
            item["fibers"] = int(a["fibers"])
        item.update(length=a.get("length"), up=bool(a.get("up", True)))
        links.append(item)
    data = {"format": "pynet2plan-topology", "version": 2, "name": G.graph.get("name", "Untitled network"),
            "nodes": nodes, "links": links}
    with open(path, "w", encoding="utf-8") as fh:            # a JSON file, readable as text
        json.dump(data, fh, indent=2, ensure_ascii=False)


def save_traffic(demands, path):
    """Saves the demands as a PyNet2Plan traffic file (Traffic > Open traffic matrix... in the tool)."""
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["id", "origin", "destination", "traffic"])          # the header line
        for d in demands:
            w.writerow([d.id, d.origin, d.destination, d.traffic])     # one line per demand


def from_networkx(H, capacity=40, name=None, size=1000.0):
    """A graph made by a NetworkX generator (for example nx.waxman_graph) as a PyNet2Plan
    network: two links per edge, node positions scaled to a size x size square, lengths
    computed from the positions."""
    pos = nx.get_node_attributes(H, "pos")           # the positions given by the generator, if any
    if len(pos) != H.number_of_nodes():
        pos = nx.spring_layout(H, seed=0)            # otherwise, positions computed for a drawing
    xs, ys = [float(pos[n][0]) for n in H], [float(pos[n][1]) for n in H]
    span = max(max(xs) - min(xs), max(ys) - min(ys)) or 1.0
    G = nx.DiGraph(name=name or f"{type(H).__name__} ({H.number_of_nodes()} nodes)")
    for n, x, y in zip(H, xs, ys):                   # node names become strings: 0 -> "0"
        G.add_node(str(n), x=round((x - min(xs)) / span * size, 1), y=round((y - min(ys)) / span * size, 1),
                   up=True)
    for u, v in H.edges():
        if u != v and not G.has_edge(str(u), str(v)):    # skip self-links and duplicates
            add_link(G, str(u), str(v), capacity=capacity, bidirectional=not H.is_directed())
    return G


# ---- the functions grouped as in PyNet2Plan, to write network.load_topology(...) and so on -----
network = types.SimpleNamespace(load_topology=load_topology, describe=describe, add_city=add_city,
                                add_link=add_link, save_topology=save_topology, from_networkx=from_networkx)
traffic = types.SimpleNamespace(load_traffic=load_traffic, total_traffic=total_traffic,
                                random_traffic=random_traffic, save_traffic=save_traffic)

# ---- the network and the traffic used in the whole notebook -------------------------------------
G = network.load_topology(TOPOLOGY)              # Spain-7, a networkx.DiGraph
D = traffic.load_traffic(TRAFFIC, G)             # the traffic of the lab, a list of 24 demands
print(network.describe(G))
print(f"{len(D)} demands, {traffic.total_traffic(D)} wavelengths")

### Setup, cell 3: two tools of the notebook

- **`run(exercise)`** runs an exercise of Part 2 on `G` and `D`. It first prints the title of the exercise, then the results printed by the exercise. If the exercise is not finished yet, it prints `not completed yet:` followed by the TODO that is missing, instead of stopping with an error. Example: `run(exercise_1)`.
- **`draw(H, paths)`** draws the network `H` on a map, with every path of the list `paths` in a different colour. A path is a list of nodes. Examples:
    - `draw(G)`: the network only,
    - `draw(G, [["Sevilla", "Madrid", "Valencia"]])`: one path (note the two pairs of brackets: a list containing one path),
    - `draw(H, [path1, path2], title="My paths")`: two paths and a title,
    - `draw(G, [path1, path2, path3], separate=True)`: one small map for each path.

`draw` places the nodes according to their latitude and longitude, so it works only for networks with geographic coordinates, such as Spain-7, also after adding a city or cutting a fibre. The code of the cell is commented, but you do not need to read it.

In [ ]:
# 3. Tools of the notebook (given): run an exercise, draw a network and some paths on a map

def run(exercise):
    """Runs an exercise on G and D, and says which TODO is still missing."""
    print(exercise.__doc__.split("\n")[0])      # the first line of the docstring: the title
    try:
        exercise(G, D)                                      # the exercise prints its own results
    except NotCompletedError as e:                          # the TODO line is still in the function
        print(f"  not completed yet: {e}")


def draw(H, paths=(), title=None, separate=False):
    """Draws H with longitude and latitude as coordinates, and each path of ``paths`` in its own
    colour. With separate=True, each path is drawn on its own map."""
    # the position of every node: x = longitude, y = latitude
    pos = {n: (H.nodes[n]["lon"], H.nodes[n]["lat"]) for n in H.nodes}
    above = {n: (x, y + 0.12) for n, (x, y) in pos.items()}         # labels above the nodes
    U = H.to_undirected()                                           # one line per fibre pair
    # the paths drawn on each map: all of them on one map, or one map per path
    groups = [[p] for p in paths] if separate and paths else [list(paths)]
    fig, axes = plt.subplots(1, len(groups), figsize=(5.2 * len(groups) if separate else 6.5, 5), squeeze=False)
    for i, (ax, group) in enumerate(zip(axes[0], groups)):
        nx.draw_networkx_edges(U, pos, ax=ax, edge_color="#9e9e9e", width=1.5)   # all links, grey
        for k, path in enumerate(group):                                # each path, in colour
            color = ("#eb6834", "#1baf7a", "#2a78d6", "#7b4fd1")[(i + k) % 4]
            nx.draw_networkx_edges(U, pos, edgelist=list(zip(path, path[1:])), ax=ax, edge_color=color,
                                   width=4, label=" > ".join(path))
        nx.draw_networkx_nodes(H, pos, ax=ax, node_size=90, node_color="#2a78d6")
        nx.draw_networkx_labels(H, above, ax=ax, font_size=11, verticalalignment="bottom")
        ax.margins(0.12)
        ax.set_axis_off()
        if separate:
            ax.set_title(" > ".join(group[0]), fontsize=10)
        elif group:                                                  # legend below the map
            ax.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), fontsize=10, frameon=False)
    if separate:
        fig.suptitle(title or H.graph.get("name", ""), fontsize=13)
    else:
        axes[0][0].set_title(title or H.graph.get("name", ""), fontsize=13)
    plt.show()
    plt.close(fig)

# Part 1. A short introduction to NetworkX

NetworkX is the Python library used by PyNet2Plan to represent networks. This part presents, one section at a time, all the calls needed in the exercises, using the network of the lab. It follows sections 1 to 7 of handout 2.

**What to do.** Run the cells in order and read their output. There is nothing to write in Part 1, but you can change the values in the cells (for example the cities of a path) and run them again to experiment. The exercises of Part 2 reuse this code, so it pays to read it carefully.

| section | you learn how to |
|---|---|
| 1. Graphs, nodes and links | build a directed graph and read the attributes of its nodes and links |
| 2. The network of the lab | explore Spain-7: links leaving and entering a node, attributes of every link |
| 3. Shortest paths | find the path with the fewest hops and the shortest path in kilometres |
| 4. The k shortest paths | list several candidate paths, from the shortest one |
| 5. Connectivity and failures | check the connectivity of a network and simulate a fibre cut |
| 6. Demands | read the traffic matrix and generate a random one from a seed |
| 7. Two bounds from the graph | compute a lower bound on the wavelengths used and on the capacity |
| 8. Generating networks | create synthetic networks with the generators of NetworkX |

A convention used in every cell: the library is imported as `nx`, so every function of NetworkX is written `nx.<name>(...)`.

## 1. Graphs, nodes and links

NetworkX represents a graph as a Python object. `nx.Graph()` is an **undirected** graph, in which an edge simply connects two nodes. `nx.DiGraph()` is a **directed** graph, in which every edge goes *from* one node *to* another.

An optical network is a directed graph. Between two cities there is usually a **pair of fibres**, one for each direction, and each direction is a separate **link** with its own capacity and its own load: the link Madrid → Sevilla may be full while Sevilla → Madrid is almost empty.

Nodes and links can carry **attributes**, that is any number of named values, stored in Python dictionaries. In PyNet2Plan the nodes have the attributes `lat` and `lon` (their coordinates), and the links have `capacity` (wavelengths per fibre), `fibers` (number of fibres) and `length` (km).

**What the next cell does.** It builds by hand a tiny network with three nodes, A, B and C, placed at the coordinates of Milan, Turin and Genoa, and three links: A → B and B → A (a fibre pair), and A → C alone. Then it prints what NetworkX has stored.

**What to look at in the output.**

- `list(T.nodes)` lists the names of the nodes, and `T.nodes["A"]` returns the dictionary of the attributes of node A.
- `list(T.edges)` lists the links as pairs `(origin, destination)`. A → B and B → A are two different links.
- `T.edges["A", "C"]` returns the attributes of the link A → C.
- `T.has_edge("C", "A")` is `False`, because only the direction A → C was added.

In [ ]:
T = nx.DiGraph(name="Tiny")                    # an empty directed graph, with a name
T.add_node("A", lat=45.46, lon=9.19)           # a node with two attributes (Milan)
T.add_node("B", lat=45.07, lon=7.69)           # (Turin)
T.add_node("C", lat=44.41, lon=8.93)           # (Genoa)
T.add_edge("A", "B", capacity=40, fibers=1, length=126.0)   # the link A -> B ...
T.add_edge("B", "A", capacity=40, fibers=1, length=126.0)   # ... and the opposite one
T.add_edge("A", "C", capacity=10, fibers=2, length=118.0)   # 2 fibres of 10 wavelengths

print("nodes:", list(T.nodes))                 # the names of the nodes
print("node A:", T.nodes["A"])                 # the attributes of node A, a dictionary
print("links:", list(T.edges))                 # every link as a pair (origin, destination)
print("link A->C:", T.edges["A", "C"])         # the attributes of the link A -> C
print("is there a link C->A?", T.has_edge("C", "A"))        # links have a direction
print(T.number_of_nodes(), "nodes and", T.number_of_edges(), "links")

**Remember.**

- The capacity of a link, in wavelengths, is the number of fibres times the wavelengths per fibre. For A → C it is `T.edges["A", "C"]["fibers"] * T.edges["A", "C"]["capacity"]` = 2 × 10 = 20 wavelengths. All the networks of the lab have one fibre per link, but the formulations always use this product.
- Asking for a link that does not exist, for example `T.edges["C", "A"]`, raises a `KeyError`. Use `T.has_edge(u, v)` to check first.
- In PyNet2Plan every network is a `DiGraph` of this kind. The function `network.add_link(G, u, v, capacity=40, bidirectional=True)` of the toolbox adds both directions at once and computes the length from the coordinates of the two nodes.

**Try it yourself.** Add the line `T.add_edge("C", "A", capacity=10, fibers=2, length=118.0)` to the cell above, run it again and look at the answer to *is there a link C->A?*

## 2. The network of the lab

`G`, loaded by the toolbox cell, is **Spain-7**: 7 cities and 8 fibre pairs, that is 16 links, with 40 wavelengths on every link. Figure 1 of handout 2 shows it on a map.

The links attached to a node are often needed, for example to write the flow conservation constraint of a node in an ILP (handout 3).

| call | result |
|---|---|
| `G.out_edges(n)` | the links **leaving** n, as pairs `(n, v)` |
| `G.in_edges(n)` | the links **entering** n, as pairs `(u, n)` |
| `G.out_degree(n)`, `G.in_degree(n)` | how many links leave n, or enter n |
| `G.nodes[n]` | the attributes of node n |
| `G.graph["name"]` | the attributes of the graph itself, here its name |

**What the next cell does.** It prints the size of the network, the attributes of the node Madrid, the links leaving and entering Madrid and the out-degree of every node.

**What to look at in the output.** Madrid has three attributes, `lat`, `lon` and `up`. The attribute `up` is `True` for a working node or link, and becomes `False` when a failure is simulated in the graphical interface. Madrid has 3 outgoing links, one towards each neighbour, and the 3 opposite links enter it.

In [ ]:
# the name and the size of the network
print(G.graph["name"], "-", G.number_of_nodes(), "nodes,", G.number_of_edges(), "links")
print("Madrid:", G.nodes["Madrid"])                        # the attributes of a node: lat, lon, up
print("links leaving Madrid:", list(G.out_edges("Madrid")))
print("links entering Madrid:", list(G.in_edges("Madrid")))
print("out-degree:", dict(G.out_degree()))                 # the number of links leaving every node

A loop over `G.edges(data=True)` gives every link together with its attributes, as triples `(u, v, attributes)`. The next cell uses it to print a table of the 16 links, sorted by name, and then draws the network with the tool `draw`.

**What to look at in the output.** Every fibre pair appears twice, once per direction, with the same length (for example Madrid → Sevilla and Sevilla → Madrid, 390.2 km). On the map each fibre pair is drawn as a single line.

In [ ]:
# the header of the table, then one row per link
print(f"{'link':22s} {'fibers':>6s} {'capacity':>8s} {'length (km)':>12s}")
for u, v, a in sorted(G.edges(data=True)):           # u = origin, v = destination, a = attributes
    print(f"{u + ' -> ' + v:22s} {a['fibers']:6d} {a['capacity']:8d} {a['length']:12.1f}")
draw(G)                                              # the map of Spain-7

**Remember.** Inside an f-string, `{value:22s}` pads a text to 22 characters, `{value:6d}` writes an integer in 6 characters, `{value:12.1f}` writes a number with one decimal in 12 characters, and `>` aligns to the right. This is how the aligned tables of the exercises are printed.

## 3. Shortest paths

"Shortest" depends on what is measured.

- `nx.shortest_path(G, s, t)` returns the path with the **fewest links (hops)** from s to t, as a list of nodes.
- `nx.shortest_path(G, s, t, weight="length")` returns the path of **minimum total length**: it adds up the attribute `length` of the links, in kilometres.
- `nx.path_weight(G, path, weight="length")` adds up an attribute along a given path, here its length in km.

The two criteria matter for different reasons. A demand uses its wavelengths on every link of its path, so a path with few hops uses few wavelengths in total, and this is what the formulations of handout 3 minimise. The length in km gives the propagation delay, and decides whether the optical signal can reach the destination without regeneration.

**What the next cell does.** From Sevilla to Valencia it computes the path with the fewest hops and the shortest path in km, prints both with their hops and kilometres, and draws them on the map.

In [ ]:
s, t = "Sevilla", "Valencia"                           # origin and destination
p_hops = nx.shortest_path(G, s, t)                     # the fewest hops (every link counts 1)
p_km = nx.shortest_path(G, s, t, weight="length")      # the fewest kilometres
for label, p in (("fewest hops", p_hops), ("shortest km", p_km)):
    # " > ".join(p) writes the path as text, and len(p) - 1 is its number of hops (links)
    print(f"{label}: {' > '.join(p)}  ({len(p) - 1} hops, {nx.path_weight(G, p, weight='length'):.0f} km)")
draw(G, [p_hops, p_km], title=f"{s} to {t}: fewest hops and shortest length")

The two criteria give different paths: the path through Madrid has one hop less, while the path along the coast is 35 km shorter. Exercise 1 asks what happens to these two paths when a fibre is cut.

Without s and t, `nx.shortest_path_length(G)` gives the number of hops between **every pair** of nodes at once. Converted with `dict(...)` it becomes a dictionary of dictionaries, in which `hops[s][t]` is the number of hops from s to t. The dictionary `hops` is used again in section 7.

In [ ]:
hops = dict(nx.shortest_path_length(G))          # hops[s][t] = number of hops from s to t
print("hops from Zaragoza:", hops["Zaragoza"])   # a dictionary: destination -> number of hops
print("hops from Sevilla to Barcelona:", hops["Sevilla"]["Barcelona"])

## 4. The k shortest paths

The route formulations of Lab 2 do not consider every possible path: they choose the path of each demand among a few **candidate paths**, usually the k shortest ones.

`nx.shortest_simple_paths(G, s, t, weight="length")` generates all the paths without loops ("simple" paths) from s to t, from the shortest to the longest. It is a *generator*, which computes the paths one at a time, only when they are requested. `islice(generator, k)` takes the first k of them without computing the others.

**What the next cell does.** It lists the 3 shortest paths from Zaragoza to Malaga by length, and draws each one on its own map (`separate=True`).

**What to look at in the output.** The paths are sorted by kilometres, not by hops: the second path has more hops than the first, and the third has the same number of hops as the second but is longer.

In [ ]:
from itertools import islice                     # islice takes the first k items of a generator

# the first k = 3 paths, from the shortest in km
paths = list(islice(nx.shortest_simple_paths(G, "Zaragoza", "Malaga", weight="length"), 3))
for i, p in enumerate(paths, 1):                 # i = 1, 2, 3 and p = the path
    print(f"path {i}: {' > '.join(p)}  ({len(p) - 1} hops, {nx.path_weight(G, p, weight='length'):.0f} km)")
draw(G, paths, title="The 3 shortest paths from Zaragoza to Malaga", separate=True)

## 5. Connectivity and failures

A directed graph is **strongly connected** when every node can reach every other node following the direction of the links. Then every demand has at least one path, if the capacity is not a concern. `nx.is_strongly_connected(G)` checks it.

The robustness of a network is studied on its undirected version, `G.to_undirected()`, in which each fibre pair becomes a single edge.

- `nx.bridges(U)` lists the edges whose failure, alone, disconnects the network.
- `nx.edge_connectivity(U)` is the smallest number of edges whose failure disconnects the network.

A **fibre cut** interrupts both directions of a fibre pair, so it is simulated by removing two links. The removal is done on a **copy** of the network, `F = G.copy()`, so that `G` stays intact for the following cells: removing the links from `G` itself would change the result of every cell run afterwards.

**What the next cell does.** It checks the connectivity of Spain-7 and looks for bridges, then cuts the fibre Madrid–Valencia on the copy `F`, and compares the fewest-hop path from Madrid to Murcia before and after the cut.

**What to look at in the output.** Spain-7 has no bridge. Its edge connectivity is the smallest number of fibre pairs that must fail together to disconnect it: keep its value in mind for exercise 2. After the cut, the path from Madrid to Murcia takes a detour through Sevilla and Malaga.

In [ ]:
U = G.to_undirected()                            # one undirected edge for every fibre pair
print("strongly connected:", nx.is_strongly_connected(G))
# an empty list counts as False in Python, so "none" is printed when there are no bridges
print("bridges:", list(nx.bridges(U)) or "none", "- edge connectivity:", nx.edge_connectivity(U))

F = G.copy()                                    # the network after the cut of Madrid-Valencia
F.remove_edges_from([("Madrid", "Valencia"), ("Valencia", "Madrid")])     # both directions
print("Madrid -> Murcia before the cut:", " > ".join(nx.shortest_path(G, "Madrid", "Murcia")))
print("Madrid -> Murcia after the cut :", " > ".join(nx.shortest_path(F, "Madrid", "Murcia")))
print("still strongly connected:", nx.is_strongly_connected(F))

**Remember.** `H = G` does **not** copy a network: it gives a second name to the same object, and removing a link from `H` would remove it from `G` too. Use `H = G.copy()`, or the helper `fiber_cut(G, u, v)` of Part 2, which returns a copy without the two links of a fibre pair.

## 6. Demands

`D`, also loaded by the toolbox cell, is the traffic matrix of the lab: a Python list of 24 demands. Each demand `d` has four fields.

| field | meaning | example |
|---|---|---|
| `d.id` | the number of the demand | `0` |
| `d.origin` | the origin node | `"Sevilla"` |
| `d.destination` | the destination node | `"Barcelona"` |
| `d.traffic` | the wavelengths to carry | `8` |

The matrix was generated with `traffic.random_traffic(G, 24, 4, 16, seed=9)`: 24 demands between random pairs of nodes, each of 4 to 16 wavelengths. A **seed** fixes the sequence of random numbers, so the same seed always gives the same demands, on every computer: anybody who knows the seed can repeat a random experiment.

**What the next cell does.** It prints the size of the matrix and its first 4 demands, then generates the matrix again from its seed and checks that the result is identical to `D`.

In [ ]:
print(len(D), "demands,", traffic.total_traffic(D), "wavelengths")
for d in D[:4]:                          # the first 4 demands (D[:4] is a slice of the list)
    print(" ", d, "- origin", d.origin, "- traffic", d.traffic)
print("random_traffic(G, 24, 4, 16, seed=9) gives the same demands:",
      traffic.random_traffic(G, 24, 4, 16, seed=9) == D)      # == compares the two lists

**Try it yourself.** In the last line of the cell above, change the seed from 9 to 10 and run the cell again: the answer becomes `False`, because another seed gives another matrix. Then set it back to 9.

## 7. Two bounds from the graph

Before solving any optimisation problem, the graph and the traffic alone give two useful numbers.

**A lower bound on the objective.** The formulations of handout 3 minimise the number of wavelengths used, summed over all the links. A demand of h wavelengths routed on a path of k hops uses h × k of them. If the capacities were unlimited, every demand would follow a path with the fewest hops, and the total would be

$$\text{lower bound} = \sum_{d \in D} h_d \times \text{hops}(s_d, t_d)$$

where $s_d$, $t_d$ and $h_d$ are the origin, the destination and the wavelengths of demand d. No routing can use fewer wavelengths, so this value is a lower bound for the optimum of the formulations. With limited capacities some demands must take longer paths, and the optimum is larger.

**What the next cell does.** It computes this sum with the dictionary `hops` of section 3.

In [ ]:
# the sum, over all the demands d, of h_d x hops(s_d, t_d)
lower = sum(d.traffic * hops[d.origin][d.destination] for d in D)
print("fewest-hop routing:", lower, "wavelengths")

**A lower bound on the capacity: the cut condition.** Take any set S of nodes. All the traffic that goes from the nodes of S to the nodes outside S must cross the links that **leave** S. If every link has C wavelengths and k links leave S, at most C × k wavelengths can leave S. Therefore

$$C \geq \left\lceil \frac{\text{traffic from } S \text{ to the other nodes}}{\text{number of links leaving } S} \right\rceil \quad \text{for every set } S$$

The brackets ⌈ ⌉ mean *rounded up* (`math.ceil` in Python), because C is a whole number of wavelengths. Every set S gives a bound, and the largest one is the most restrictive. Two calls help.

- `nx.edge_boundary(G, S)` lists the links leaving S, as pairs `(u, v)` with u in S and v outside S.
- `itertools.combinations(G.nodes, size)` gives every set of `size` nodes, so a loop over size = 1, ..., 6 visits all the 2⁷ − 2 = 126 sets of nodes that are neither empty nor the whole network.

**What the next cell does.** It first evaluates the bound for one set, all the nodes except Malaga and Murcia. Then it repeats the computation for all the 126 sets and keeps the largest bound. Finally it draws the two links that enter Malaga and Murcia.

**What to look at in the output.** 68 wavelengths must reach Malaga and Murcia, and only 2 links enter them, so every link needs at least ⌈68 / 2⌉ = 34 wavelengths, and no other set gives a larger bound. Handout 3 confirms that the splittable flow formulation is infeasible for every capacity below 34.

In [ ]:
# One set S: all the nodes except Malaga and Murcia
S = {"Madrid", "Barcelona", "Valencia", "Zaragoza", "Sevilla"}     # all but Malaga and Murcia
links = list(nx.edge_boundary(G, S))                               # the links leaving S
need = sum(d.traffic for d in D if d.origin in S and d.destination not in S)    # traffic leaving S
print(f"{need} wavelengths leave S on the links {links}: C >= {math.ceil(need / len(links))}")

# Every set S: keep the largest bound and its set. In a strongly connected network at least one
# link leaves every set S, so the division is always possible.
best = None                                      # (bound, set) of the most restrictive set so far
for size in range(1, G.number_of_nodes()):       # sets of 1 to 6 nodes
    for S in itertools.combinations(G.nodes, size):          # every set of `size` nodes
        bound = math.ceil(sum(d.traffic for d in D if d.origin in S and d.destination not in S)
                          / len(list(nx.edge_boundary(G, S))))
        if best is None or bound > best[0]:      # a larger bound: remember it with its set
            best = (bound, sorted(S))
print("most restrictive set:", best[1], "-> C >=", best[0])
draw(G, [["Sevilla", "Malaga"], ["Valencia", "Murcia"]], title="The two links entering Malaga and Murcia")

## 8. Generating networks

Besides real networks it is often useful to test an algorithm on many synthetic ones. NetworkX has generators for many families of graphs.

| generator | graph |
|---|---|
| `nx.cycle_graph(n)` | a ring of n nodes |
| `nx.grid_2d_graph(rows, cols)` | a grid |
| `nx.gnm_random_graph(n, m, seed=s)` | n nodes and m edges placed at random |
| `nx.waxman_graph(n, beta=b, alpha=a, seed=s)` | n nodes placed at random in a square, connected with a probability that decreases with their distance, as happens in real fibre networks |

In the Waxman model a larger `beta` gives more edges, and a larger `alpha` makes long edges more likely. The generated graphs are **undirected** and may be **disconnected**: `nx.is_connected(H)` checks it for an undirected graph, as `nx.is_strongly_connected` does for a directed one. The function `network.from_networkx` of the toolbox turns a generated graph into a network of PyNet2Plan, with two links per edge and lengths computed from the positions of the nodes. The result can be saved with `network.save_topology` and opened in the tool.

**What the next cell does.** It generates four graphs, draws them side by side with the information on their connectivity, and converts the Waxman graph into a PyNet2Plan network.

**What to look at in the output.** The title of each drawing gives the number of nodes and edges and says whether the graph is connected. The converted Waxman network has twice as many links as the generated graph has edges, one for each direction.

In [ ]:
generated = {"ring": nx.cycle_graph(8), "grid": nx.grid_2d_graph(3, 4),
             "Waxman": nx.waxman_graph(12, beta=0.7, alpha=0.35, seed=7),
             "random": nx.gnm_random_graph(10, 16, seed=3)}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))          # four drawings side by side
for ax, (kind, H) in zip(axes, generated.items()):
    # the Waxman generator stores the position of every node in the attribute "pos", the others
    # do not, and spring_layout computes positions for a drawing
    pos = nx.get_node_attributes(H, "pos") or nx.spring_layout(H, seed=0)
    nx.draw(H, pos, ax=ax, node_size=60, node_color="#2a78d6", edge_color="#9e9e9e")
    ax.set_title(f"{kind}: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges\n"
                 f"connected: {nx.is_connected(H)}", fontsize=11)
plt.show()
plt.close(fig)
W = network.from_networkx(generated["Waxman"], capacity=40, name="Waxman 12")    # 2 links per edge
print(network.describe(W))

## End of Part 1

You have now seen all the calls needed in Part 2. In short:

- a network is a `networkx.DiGraph`, whose links are pairs `(u, v)` with the attributes `capacity`, `fibers` and `length`,
- `nx.shortest_path` finds the path with the fewest hops, or the shortest in km with `weight="length"`,
- a fibre cut is simulated by removing two links **from a copy** of the network,
- the traffic is a list of demands with `origin`, `destination` and `traffic`, and a seed makes a random matrix repeatable,
- the graph alone gives a lower bound on the wavelengths used (408 for the lab traffic) and, with the cut condition, a lower bound on the capacity (34 wavelengths per link).

The complete tour, with more comments, is the script `labs/lab1_networkx_tour.py` described in handout 2.

# Part 2. Exercises

Each exercise is a Python function, `exercise_1(G, D)` to `exercise_7(G, D)`. It receives the network `G` (Spain-7) and the demands `D` (the traffic of the lab), which `run` passes to it, and it **prints** its results: it does not need to return anything. All the modules (`nx`, `math`, `itertools`, `os`) and the names `network`, `traffic`, `ROOT` and `OUT` defined by the setup cells can be used inside the functions.

## How to work on an exercise

1. Read the text of the exercise, in the cell above its function: the question, the steps, and the format of the expected output.
2. In the function, write your code **in place of** the line `raise NotCompletedError("TODO ...")`, which you delete. The comments inside the function repeat the steps.
3. Run the cell of the function. Nothing is printed, because the cell only *defines* the function.
4. Run the cell `run(exercise_n)` below it, which *executes* the function and prints its results.
5. If something is wrong, change the function and repeat steps 3 and 4. Always run the cell of the function again after a change, otherwise `run` executes the old version.

While the line `raise NotCompletedError(...)` is still in a function, `run` prints `not completed yet:` followed by the TODO that is missing. This is normal for the exercises you have not done yet.

## Advice

- **Never modify `G` or `D`.** Every exercise uses them. To change the network, work on a copy: `H = G.copy()`, or `H = fiber_cut(G, u, v)`. Writing `H = G` does **not** copy the network.
- **Reuse Part 1.** Most of the code you need is already in a cell of Part 1: find the cell that does something similar and adapt it.
- **Build the solution step by step.** Write a few lines, run them, look at what they print, then continue. When a result looks strange, print the intermediate values (a path, a set, a number) to see where the problem is.
- **Indentation.** The code inside the function is indented by 4 spaces, and the code inside a `for` or an `if` by 4 more.
- **Output.** Start every printed line with two spaces, as in the examples of the exercises, so that your output is easy to compare with the one discussed at the end of the lab.

## If something goes wrong

| message | cause | what to do |
|---|---|---|
| `NameError: name 'G' is not defined` (or `nx`, `network`, `traffic`, `run`) | the setup cells have not been run, or the session was restarted | run the setup cells again (*Runtime > Run before* in Colab) |
| `NameError: name 'fiber_cut' is not defined` | the cell of the helpers has not been run | run the cell of the helpers, below |
| `not completed yet: TODO n ...` | the line `raise NotCompletedError(...)` is still in the function, or the cell of the function was not run again after the change | delete the line, run the cell of the function, then the cell `run(...)` |
| `IndentationError` | the code inside the function is not indented correctly | indent it by 4 spaces, and by 4 more inside a `for` or an `if` |
| `KeyError: 'Madrid'` or `KeyError: ('Madrid', 'Valencia')` | a node or a link that does not exist, for example after a cut | check the names, and use `H.has_edge(u, v)` |
| `NetworkXNoPath` | there is no path between two nodes, because the network is disconnected | check `nx.is_strongly_connected(H)` before computing the paths |
| `KeyError: 'lon'` from `draw` | the network has no geographic coordinates, for example a generated network | draw it with `nx.draw`, as in Part 1, section 8 |
| `ModuleNotFoundError: No module named 'networkx'` (Jupyter or VS Code) | the notebook does not use the Python of the lab | select the kernel of the virtual environment `.venv` of the lab |

## Helpers (given)

The next cell defines four helper functions, the same as in `labs/lab1_exercises.py`. Run it once: the exercises use them.

| helper | returns | example |
|---|---|---|
| `fiber_cut(G, u, v)` | a **copy** of G without the two links of the fibre pair u–v. G is not modified | `H = fiber_cut(G, "Madrid", "Valencia")` |
| `fibers(G)` | the list of the fibre pairs of G, each once, as pairs `(u, v)` | `[("Madrid", "Zaragoza"), ...]`, 8 pairs for Spain-7 |
| `fewest_hop_bound(G, D)` | the lower bound of Part 1, section 7: the wavelengths used if every demand of D follows a fewest-hop path in G | `fewest_hop_bound(G, D)` is 408 |
| `km(G, path)` | the length of a path, in km | `km(G, ["Sevilla", "Madrid"])` |

`fewest_hop_bound` needs a path for every demand: on a disconnected network it stops with a `KeyError`. The cell after the helpers shows what they return.

In [ ]:
def fiber_cut(G, u, v):
    """A copy of G without the two directions of the link u-v (a fibre cut)."""
    H = G.copy()                                 # a copy: G itself is not modified
    H.remove_edges_from([(u, v), (v, u)])        # both directions of the fibre pair
    return H


def fibers(G):
    """Every pair of opposite links once: ('Madrid', 'Zaragoza'), ..."""
    seen = []
    for u, v in G.edges:                         # every link (u, v) of the network
        if (u, v) not in seen and (v, u) not in seen:     # its fibre pair is not in the list yet
            seen.append((u, v))
    return seen


def fewest_hop_bound(G, D):
    """Wavelengths used if every demand follows a path with the fewest hops (handout 2, section 7)."""
    hops = dict(nx.shortest_path_length(G))      # hops[s][t] for every pair of nodes
    # (a KeyError if a demand has no path, in a disconnected network)
    return sum(d.traffic * hops[d.origin][d.destination] for d in D)


def km(G, path):
    """The length of a path in km: the sum of the attribute length of its links."""
    return nx.path_weight(G, path, weight="length")

In [ ]:
# What the helpers return: compare each line of code with what it prints
print("fibre pairs of Spain-7:", len(fibers(G)))
print("  ", fibers(G))
H = fiber_cut(G, "Madrid", "Valencia")      # a copy of G without the fibre pair Madrid-Valencia
print("links of G:", G.number_of_edges(), "- links of the copy:", H.number_of_edges())
print("G still has the link Madrid -> Valencia:", G.has_edge("Madrid", "Valencia"))
print("fewest-hop bound of G:", fewest_hop_bound(G, D), "wavelengths")
print("length of Sevilla > Madrid:", km(G, ["Sevilla", "Madrid"]), "km")

## Python reminders

The exercises use a few constructions of Python that are worth recalling. The next cell shows each of them at work: run it and compare each line of code with what it prints.

| construction | example | result |
|---|---|---|
| a path as text | `" > ".join(["A", "B", "C"])` | `"A > B > C"` |
| the hops of a path | `len(path) - 1` | a path of 3 nodes has 2 hops |
| the intermediate nodes | `path[1:-1]` | all the nodes except the first and the last |
| a counter for every node | `count = {n: 0 for n in G.nodes}` | a dictionary with the value 0 for every node |
| sorting by value | `sorted(count.items(), key=lambda kv: kv[1], reverse=True)` | the pairs (node, value), largest value first |
| rounding up | `math.ceil(7 / 2)` | 4 |
| every set of k nodes | `itertools.combinations(nodes, k)` | for k = 2, every pair of nodes |
| the maximum while looping | `if best is None or value > best[1]: best = (name, value)` | the largest value seen so far, with its name |
| a formatted line | `f"{name:10s} {value:5d}"` | a text padded to 10 characters and an integer in 5 |
| a copy of the network | `H = G.copy()` | H can be changed without changing G |

In [ ]:
# Python reminders: each print shows the result of the construction written in its line
path = ["Sevilla", "Madrid", "Valencia"]                       # a path is a list of nodes
print("path as text      :", " > ".join(path))
print("hops              :", len(path) - 1)                    # 3 nodes, 2 links
print("intermediate nodes:", path[1:-1])                       # without the first and the last node

count = {n: 0 for n in ["A", "B", "C"]}                        # a counter for every node, all at 0
count["B"] += 7                                                # adds 7 to the counter of B
count["C"] += 3
print("counters          :", count)
print("sorted by value   :", sorted(count.items(), key=lambda kv: kv[1], reverse=True))

print("rounding up       :", math.ceil(7 / 2), "and", math.ceil(8 / 2))
print("pairs of nodes    :", list(itertools.combinations(["A", "B", "C"], 2)))

best = None                                                    # the maximum while looping
for name, value in count.items():
    if best is None or value > best[1]:
        best = (name, value)
print("largest counter   :", best)

print(f"formatted line    : |{'Madrid':10s}|{42:5d}|{3.14159:6.2f}|")      # widths and decimals

H = G.copy()                                                   # a copy of the network ...
H.remove_edge("Madrid", "Valencia")                            # ... can be changed ...
print("G has Madrid -> Valencia:", G.has_edge("Madrid", "Valencia"),
      "- the copy has it:", H.has_edge("Madrid", "Valencia"))  # ... without changing G

## The NetworkX calls you need

All of them are explained in Part 1 and in handout 2, and used in `labs/lab1_networkx_tour.py`.

| call | result | see |
|---|---|---|
| `nx.shortest_path(G, s, t)` | the path with the fewest hops, as a list of nodes | section 3 |
| `nx.shortest_path(G, s, t, weight="length")` | the shortest path in kilometres | section 3 |
| `nx.shortest_path_length(G, s, t)` | the number of hops of the fewest-hop path | section 3 |
| `dict(nx.shortest_path_length(G))` | the hops between every pair of nodes, `hops[s][t]` | section 3 |
| `nx.path_weight(G, path, weight="length")` | the length of a path in km (helper `km(G, path)`) | section 3 |
| `nx.is_strongly_connected(H)` | `True` if every node can reach every other node | section 5 |
| `G.copy()`, `H.remove_edges_from([(u, v), (v, u)])` | a copy of the network, and a fibre cut on the copy | section 5 |
| `G.nodes`, `G.edges`, `G.edges[u, v]["length"]` | the nodes, the links and an attribute of a link | sections 1 and 2 |
| `G.out_edges(n)`, `G.in_edges(n)` | the links leaving and entering node n | section 2 |
| `nx.edge_boundary(G, S)` | the links leaving the set of nodes S | section 7 |
| `itertools.combinations(nodes, k)` | every set of k nodes | section 7 |
| `nx.waxman_graph(n, beta=b, alpha=a, seed=s)`, `nx.is_connected(W)` | a random geographic graph, and its connectivity | section 8 |
| `nx.diameter(H)` | the largest number of hops of a fewest-hop path, over all the pairs of nodes | exercise 7 |
| `d.origin`, `d.destination`, `d.traffic` | the fields of a demand of `D` | section 6 |

The files mentioned in the exercises (`data/...`) are those embedded in the first setup cell, and `network.*` and `traffic.*` are the functions of the toolbox. The tool `draw(H, [path1, path2])` shows a network and some paths on a map (`separate=True`: one map per path).

## Exercise 1. Paths before and after a fibre cut

**Question.** Paths from Sevilla to Valencia with the fewest hops and in kilometres, before and after the cut of the fibre Madrid–Valencia. How many hops and kilometres does the cut add?

**Why it matters.** When a fibre is cut, the demands that used it must be rerouted on the remaining links. A path with more hops uses more wavelengths, one on each link, and a longer path in km adds delay. The question is how much worse the best paths become.

**Steps.**

1. Build the network after the cut with `fiber_cut(G, "Madrid", "Valencia")`. The network before the cut is `G` itself.
2. For each of the two networks:
    - compute the path with the fewest hops from Sevilla to Valencia, with `nx.shortest_path`,
    - compute the shortest path in km, with `nx.shortest_path` and `weight="length"`,
    - print each path with its number of hops and its length in km.
3. Compare the values before and after the cut, for each of the two criteria.

A `for` loop over the two networks avoids writing the same code twice, for example `for label, H in (("before the cut", G), ("after the cut ", H_cut)):`, where `H_cut` is the network after the cut.

**Functions.** `fiber_cut`, `nx.shortest_path`, `km(H, path)`, `" > ".join(path)`, `len(path) - 1`. The cell of Part 1, section 3 computes the two paths for G: start from it.

**Expected output.** The first line is printed by `run`, and the dots stand for your paths and values.

```text
Paths from Sevilla to Valencia, before and after the cut of the fibre Madrid-Valencia.
  before the cut: fewest hops Sevilla > ... > Valencia (. hops, ... km)
                  shortest km Sevilla > ... > Valencia (. hops, ... km)
  after the cut : fewest hops Sevilla > ... > Valencia (. hops, ... km)
                  shortest km Sevilla > ... > Valencia (. hops, ... km)
```

**Check your result.**

- Before the cut you must find the two paths of Part 1, section 3.
- After the cut neither path can be better than before: removing links never makes a shortest path shorter.
- To see the paths after the cut, add `draw(H_cut, [path1, path2])` inside your function.

**Think about it.** Which of the two criteria is affected by the cut, and why? Does the shortest path in km use the fibre Madrid–Valencia?

In [ ]:
def exercise_1(G, D):
    """Paths from Sevilla to Valencia, before and after the cut of the fibre Madrid-Valencia."""
    # ---- TODO 1 ----
    # Goal: print the fewest-hop path and the shortest path in km from Sevilla to Valencia, with
    # their hops and km, in the network before the cut (G) and after the cut of Madrid-Valencia.
    #
    # Steps:
    #   1. the network after the cut: fiber_cut(G, "Madrid", "Valencia")   (G is not modified)
    #   2. for each network, before and after the cut:
    #        a. the fewest hops:  nx.shortest_path(network, "Sevilla", "Valencia")
    #        b. the shortest km:  nx.shortest_path(network, "Sevilla", "Valencia", weight="length")
    #        c. print each path (" > ".join(path)), its hops (len(path) - 1)
    #           and its length (km(network, path))
    #   3. compare: how many hops and km does the cut add, for each criterion?
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 1: paths before and after the cut")

In [ ]:
run(exercise_1)                # runs exercise 1 and prints its results

## Exercise 2. The most harmful single fibre cut

**Question.** For each of the 8 single fibre cuts: does the network remain connected, and how many wavelengths does the fewest-hop routing of the lab traffic need? Which cut is the most harmful?

**Why it matters.** A network planner checks every possible single failure. Here the damage of a cut is measured with the lower bound of Part 1, section 7, that is the wavelengths used if every demand follows a fewest-hop path in the damaged network. The larger its increase with respect to the intact network, the more the cut pushes the traffic onto longer paths. The capacity is ignored, as if every link were large enough: handout 3, section 5 repeats the experiment with 60 wavelengths per link.

**Steps.**

1. Compute the bound of the intact network, `base = fewest_hop_bound(G, D)`. It is 408, as in Part 1.
2. Loop over the 8 fibre pairs: `for u, v in fibers(G):`
3. For each pair, build the damaged network `H = fiber_cut(G, u, v)`.
4. Check whether `H` is still strongly connected. If it is not, print that the cut disconnects the network and go on with the next pair (`continue`): some demands would have no path, and the bound could not be computed.
5. Otherwise compute `b = fewest_hop_bound(H, D)` and print it together with the increase `b - base`.
6. Remember the cut with the largest bound, and print it after the loop.

**Functions.** `fibers`, `fiber_cut` and `fewest_hop_bound` (helpers), `nx.is_strongly_connected`.

**Expected output** (one line for each of the 8 cuts):

```text
The effect of every single fibre cut on the fewest-hop bound of the lab traffic.
  no cut: 408 wavelengths
  cut Madrid-Zaragoza: ... wavelengths (+..)
  cut ...
  most harmful cut: ...-... (+... wavelengths)
```

**Check your result.**

- You must print 8 cuts, one for each fibre pair, and every increase must be zero or positive: a cut never makes a path shorter.
- Is your answer on connectivity consistent with the edge connectivity printed in Part 1, section 5?

**Think about it.** Why does the most harmful cut increase the bound so much? Look at the map: which demands lose their shortest path, and how long are their detours?

In [ ]:
def exercise_2(G, D):
    """The effect of every single fibre cut on the fewest-hop bound of the lab traffic.

    For each cut: is the network still connected, and how many wavelengths does the
    fewest-hop routing of the lab traffic need?"""
    # ---- TODO 2 ----
    # Goal: cut each of the 8 fibre pairs in turn, and compute the fewest-hop bound of the lab
    # traffic in the damaged network. Which cut increases the bound the most?
    #
    # Steps:
    #   1. base = fewest_hop_bound(G, D)        (408, the intact network)
    #   2. for u, v in fibers(G):               (the 8 fibre pairs)
    #        a. H = fiber_cut(G, u, v)
    #        b. if not nx.is_strongly_connected(H): print that the cut disconnects the network,
    #           then continue with the next pair
    #        c. compute fewest_hop_bound(H, D) and print it with its increase over base
    #        d. if it is the largest bound seen so far, remember it together with (u, v)
    #   3. after the loop, print the most harmful cut and its increase
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 2: fewest-hop bound after every single fibre cut")

In [ ]:
run(exercise_2)                # runs exercise 2 and prints its results

## Exercise 3. The cut bound for another traffic matrix

**Question.** The cut bound of section 5 of handout 2 (section 7 of this notebook) for the other traffic matrix, `data/traffic/spain7_traffic.csv`: which set S is the most restrictive, and which capacity does it require?

**Why it matters.** The cut bound gives, from the traffic matrix alone and without solving any optimisation problem, the smallest capacity C for which a routing *may* exist. With a different traffic matrix the bottleneck of the network may be somewhere else.

**Steps.**

1. Load the other matrix, `D2 = traffic.load_traffic(os.path.join(ROOT, "data", "traffic", "spain7_traffic.csv"), G)`, and print its number of demands and its total traffic.
2. For every set S of 1 to 6 nodes, compute:
    - the traffic from S to the other nodes, that is the sum of `d.traffic` over the demands of `D2` whose origin is in S and whose destination is not,
    - the number of links leaving S, that is the links `(u, v)` of G with u in S and v not in S,
    - the bound, the traffic divided by the number of links and **rounded up**.
3. Keep the set with the largest bound, and print it with the other nodes (its complement), the traffic, the number of links and the bound.

**Functions.** `itertools.combinations(G.nodes, size)`, `math.ceil`, and `nx.edge_boundary(G, S)` or a count over `G.edges`. The second half of the cell of Part 1, section 7 does exactly this computation for D: copy it and adapt it to D2.

**Expected output:**

```text
Cut bound (handout 2, section 5) for the other traffic matrix of Spain-7.
  42 demands, ... wavelengths
  tightest cut: .. λ from [...] to [...] over . links  ->  C >= ..
```

**Check your result.**

- Run your code with `D` first: it must give 34, the value of Part 1, section 7. Then use `D2`.
- The loops examine 126 sets, and the bound is a whole number.
- `D2` has 42 demands: it is a full mesh, with one demand for every ordered pair of the 7 nodes (7 × 6 = 42).

**Think about it.** Compare the most restrictive set for `D2` with the one for `D`: why is it a different set? Does the cut bound guarantee that a routing exists with that capacity, or is it only a necessary condition? A sweep in the graphical interface, with this traffic matrix, answers the question.

In [ ]:
def exercise_3(G, D):
    """Cut bound (handout 2, section 5) for the other traffic matrix of Spain-7."""
    # ---- TODO 3 ----
    # Goal: the most restrictive cut condition for data/traffic/spain7_traffic.csv.
    #
    # Steps:
    #   1. D2 = traffic.load_traffic(os.path.join(ROOT, "data", "traffic", "spain7_traffic.csv"), G)
    #      and print len(D2) and traffic.total_traffic(D2)
    #   2. for every set S of 1 to 6 nodes (itertools.combinations(G.nodes, size)):
    #        traffic leaving S = sum of d.traffic over the demands d of D2
    #                            with d.origin in S and d.destination not in S
    #        links leaving S   = number of links (u, v) of G with u in S and v not in S
    #        bound             = math.ceil(traffic leaving S / links leaving S)
    #   3. keep the largest bound with its set S, then print S, the other nodes, the traffic,
    #      the number of links and the bound
    # Tip: Part 1, section 7 does the same computation for D.
    #      Run your code with D first: it must give 34.
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 3: cut bound for spain7_traffic.csv")

In [ ]:
run(exercise_3)                # runs exercise 3 and prints its results

## Exercise 4. Transit traffic of every node

**Question.** With the fewest-hop routing of the lab traffic, how many wavelengths cross each node in transit (neither origin nor destination)?

**Why it matters.** A node carries in transit the wavelengths of the demands that pass through it without starting or ending there. The node has to switch this traffic, optically or electronically, so the nodes with much transit traffic need more switching equipment, and their failure affects many demands.

**Example.** A demand of 5 wavelengths routed on Sevilla > Madrid > Valencia adds 5 wavelengths to the transit traffic of Madrid only: Sevilla is its origin and Valencia its destination.

**Steps.**

1. Create a counter for every node, all at zero: `transit = {n: 0 for n in G.nodes}`.
2. For every demand `d` of `D`:
    - find its fewest-hop path, `path = nx.shortest_path(G, d.origin, d.destination)`,
    - add `d.traffic` to the counter of every intermediate node of the path, `path[1:-1]`.
3. Print the nodes from the most to the least loaded.

**Functions.** `nx.shortest_path`, the slice `path[1:-1]`, and `sorted(transit.items(), key=lambda kv: kv[1], reverse=True)` (see the Python reminders).

**Expected output** (one line for each of the 7 nodes):

```text
Transit traffic of every node with the fewest-hop routing of the lab traffic.
  <node>      .. λ in transit
  ...
```

**Check your result.** The transit traffic of all the nodes must add up to 408 − 229 = 179 wavelengths. The reason is that a demand of h wavelengths on a path of k hops crosses k − 1 intermediate nodes, so it contributes h × (k − 1). Summed over all the demands, this gives the fewest-hop bound (408) minus the total traffic (229).

**Note.** When several paths have the same number of hops, `nx.shortest_path` returns one of them. Another choice would change the values of some nodes, but not their total.

**Think about it.** Which node would you protect first? Is the node with the most transit traffic also the node with the most links?

In [ ]:
def exercise_4(G, D):
    """Transit traffic of every node with the fewest-hop routing of the lab traffic.

    Transit = wavelengths of the demands whose path crosses the node, without starting
    or ending there."""
    # ---- TODO 4 ----
    # Goal: for every node, the wavelengths of the demands that cross it in transit when every
    # demand follows its fewest-hop path.
    #
    # Steps:
    #   1. transit = {n: 0 for n in G.nodes}            (a counter for every node)
    #   2. for every demand d in D:
    #        path = nx.shortest_path(G, d.origin, d.destination)
    #        add d.traffic to the counter of every node of path[1:-1] (the intermediate nodes)
    #   3. print the nodes sorted from the most to the least loaded
    # Check: the counters must add up to 408 - 229 = 179.
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 4: transit traffic of every node")

In [ ]:
run(exercise_4)                # runs exercise 4 and prints its results

## Exercise 5. A new city: Bilbao (Spain-8)

**Question.** Add Bilbao, linked to Madrid and Zaragoza with 40 wavelengths, generate 30 demands of 4 to 16 wavelengths with seed 3, and save both files. Open them in the graphical interface and run the splittable flow formulation. Is the problem feasible? Find the smallest feasible capacity with a sweep, and compare it with the cut bound of the new network with its traffic, computed as in exercise 3.

**Why it matters.** Networks grow: a new city is connected to the existing ones, and new traffic appears. This exercise combines the two tools of the lab: NetworkX to build the new network and its traffic, and the graphical interface of PyNet2Plan to solve the routing problem on it.

**Part A, in the notebook.**

1. Copy the network with `H = G.copy()`, and give it a name with `H.graph["name"] = "Spain-8"`.
2. Add the node with `network.add_city(H, "Bilbao", allow_online=False)`. Its coordinates are in the dictionary `CITIES` of the first setup cell.
3. Add two fibre pairs of 40 wavelengths, Bilbao–Madrid and Bilbao–Zaragoza, with `network.add_link(H, "Bilbao", "Madrid", capacity=40, bidirectional=True)` and the same call for Zaragoza. The lengths are computed from the coordinates.
4. Generate the traffic, `D8 = traffic.random_traffic(H, 30, 4, 16, seed=3)`.
5. Save the two files in the folder `OUT`. First create it with `os.makedirs(OUT, exist_ok=True)`, then call `network.save_topology(H, os.path.join(OUT, "spain8.json"))` and `traffic.save_traffic(D8, os.path.join(OUT, "spain8_traffic.csv"))`.
6. Print a short summary: `network.describe(H)`, the lengths of the two new links (for example `H.edges["Bilbao", "Madrid"]["length"]`), the number of demands and of wavelengths of `D8`, and its fewest-hop bound `fewest_hop_bound(H, D8)`.

**Part B, in PyNet2Plan on your computer.**

1. In Colab, download the two files: open the *Files* panel on the left, then the folder `results/lab1_exercises`, right click on each file and choose *Download*. In Jupyter or VS Code they are already in the folder `results/lab1_exercises` next to the notebook.
2. In PyNet2Plan open them with *File > Open topology...* and *Traffic > Open traffic matrix...*.
3. Run the splittable flow formulation. Is the problem feasible with 40 wavelengths on every link?
4. With *Compare (sweep)...* find the smallest capacity, the same on all the links, for which the problem is feasible.
5. Compute the cut bound of Spain-8 with its traffic: run your code of exercise 3 with `H` and `D8` in place of `G` and `D2`. Do the two values coincide?

**Expected output of Part A:**

```text
A new city, Bilbao, linked to Madrid and Zaragoza: topology and traffic saved for the GUI.
  Spain-8: 8 nodes, 20 links (20 up), capacity 40 λ
  Bilbao-Madrid: ... km
  Bilbao-Zaragoza: ... km
  traffic: 30 demands, ... wavelengths, fewest-hop bound ...
  saved results/lab1_exercises/spain8.json and results/lab1_exercises/spain8_traffic.csv
```

**Check your result.** Spain-8 has 8 nodes and 20 links: the 16 links of Spain-7 plus two fibre pairs. The traffic depends only on the seed and on the order of the nodes: generated in the graphical interface with the same values (*Traffic > Generate random traffic...*, 30 demands of 4 to 16 wavelengths, seed 3), it must be identical to yours.

**Think about it.** Bilbao is in the north, but it also sends traffic to the cities of the south. What happens to the bottleneck found in Part 1, section 7?

In [ ]:
def exercise_5(G, D):
    """A new city, Bilbao, linked to Madrid and Zaragoza: topology and traffic saved for the GUI."""
    # ---- TODO 5 ----
    # Goal: build Spain-8 (Spain-7 plus Bilbao), generate its traffic and save both files, to be
    # opened in the graphical interface of PyNet2Plan.
    #
    # Steps:
    #   1. H = G.copy() and H.graph["name"] = "Spain-8"
    #   2. network.add_city(H, "Bilbao", allow_online=False)
    #   3. network.add_link(H, "Bilbao", "Madrid", capacity=40, bidirectional=True), and the same
    #      for Zaragoza
    #   4. D8 = traffic.random_traffic(H, 30, 4, 16, seed=3)
    #   5. os.makedirs(OUT, exist_ok=True), then save the two files with
    #      network.save_topology(H, os.path.join(OUT, "spain8.json")) and
    #      traffic.save_traffic(D8, os.path.join(OUT, "spain8_traffic.csv"))
    #   6. print network.describe(H), the lengths of the two new links, the size of D8 and
    #      fewest_hop_bound(H, D8)
    # Then open the two files in PyNet2Plan (Part B in the text above).
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 5: Spain-8 with Bilbao")

In [ ]:
run(exercise_5)                # runs exercise 5 and prints its results

## Exercise 6 (optional). A connected Waxman network

**Question.** Generate the first connected Waxman network with 15 nodes (Part 1, section 8) and open it in the graphical interface.

**Why it matters.** A random generator may produce a disconnected graph, which is useless as a network because some demands could never be routed. A common practice is to try the seeds 0, 1, 2, ... in order and to keep the first connected graph. Since the seeds are tried in a fixed order, everybody obtains the same network.

**Steps.**

1. For `seed` = 0, 1, 2, ..., generate `W = nx.waxman_graph(15, beta=0.6, alpha=0.3, seed=seed)` and stop the loop (`break`) at the first graph for which `nx.is_connected(W)` is `True`.
2. Convert it: `H = network.from_networkx(W, capacity=40, name=f"Waxman-15 (seed {seed})")`.
3. Save it in `OUT` as `waxman15.json`, creating the folder first as in exercise 5, and print the seed and `network.describe(H)`.
4. Open the file in PyNet2Plan (*File > Open topology...*), generate a traffic matrix and run the splittable flow formulation.

Use exactly these values of `beta` and `alpha`, so that your network is the same as that of the other students.

**Functions.** `nx.waxman_graph`, `nx.is_connected` (for the undirected graph W), `network.from_networkx`, `network.save_topology`.

**Expected output:**

```text
(Optional) A connected Waxman network with 15 nodes, saved for the GUI.
  first connected network: seed .., Waxman-15 (seed ..): 15 nodes, .. links (.. up), capacity 40 λ
  saved results/lab1_exercises/waxman15.json
```

**Check your result.** The converted network has two links for every edge of W. Do not use `draw` for this network: its nodes have the positions `x` and `y` but no latitude and longitude. To see it, use `nx.draw(W, nx.get_node_attributes(W, "pos"))` followed by `plt.show()`, as in Part 1, section 8.

In [ ]:
def exercise_6(G, D):
    """(Optional) A connected Waxman network with 15 nodes, saved for the GUI."""
    # ---- TODO 6 ----
    # Goal: the first connected Waxman network with 15 nodes, saved as waxman15.json in OUT.
    #
    # Steps:
    #   1. for seed in range(100): W = nx.waxman_graph(15, beta=0.6, alpha=0.3, seed=seed),
    #      and break out of the loop as soon as nx.is_connected(W) is True
    #   2. H = network.from_networkx(W, capacity=40, name=f"Waxman-15 (seed {seed})")
    #   3. os.makedirs(OUT, exist_ok=True), then
    #      network.save_topology(H, os.path.join(OUT, "waxman15.json"))
    #   4. print the seed and network.describe(H)
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 6: a connected Waxman network")

In [ ]:
run(exercise_6)                # runs exercise 6 and prints its results

## Exercise 7 (optional). The size of some real networks

**Question.** Number of nodes and links, links per node and diameter in hops of Spain-7, NSFNET-14, COST239 and GEANT2.

**Why it matters.** These quantities describe a topology at a glance. The **links per node**, the number of links divided by the number of nodes, measure how meshed the network is. When every link has its opposite one, as in these networks, this is also the average number of fibre pairs of a node. The **diameter** is the largest number of hops of a fewest-hop path, over all the pairs of nodes: a large diameter means long paths, and therefore many wavelengths used by each demand.

**Steps.**

1. The four files are `data/topologies/spain7.json`, `data/topologies/nsfnet14.json`, `data/topologies/cost239.json` and `data/topologies/real/top22_geant2.json`. Build their names with `os.path.join(ROOT, "data", "topologies", ...)`, as in the toolbox cell, or write them directly.
2. For each file, load the network with `H = network.load_topology(path)`, then compute its name `H.graph["name"]`, `H.number_of_nodes()`, `H.number_of_edges()`, the links per node and `nx.diameter(H)`.
3. Print one row for each network, aligned in columns with f-strings.

**Expected output:**

```text
(Optional) Size of some real networks: nodes, links, average out-degree, diameter in hops.
  network                      nodes links degree diameter
  Spain-7                          7    16   2.29        .
  ...
```

**Check your result.** For Spain-7 the links per node are 16 / 7 = 2.29.

**Think about it.** Which network is the most meshed, and which one has the longest paths? How are the diameter, the number of nodes and the links per node related?

In [ ]:
def exercise_7(G, D):
    """(Optional) Size of some real networks: nodes, links, average out-degree, diameter in hops."""
    # ---- TODO 7 ----
    # Goal: a table with the name, the nodes, the links, the links per node and the diameter of
    # four networks.
    #
    # Steps:
    #   1. the files: spain7.json, nsfnet14.json and cost239.json in data/topologies, and
    #      top22_geant2.json in data/topologies/real
    #   2. for each file: H = network.load_topology(path), then H.number_of_nodes(),
    #      H.number_of_edges(), links per node = links / nodes, and nx.diameter(H)
    #   3. print one aligned row per network, for example with
    #      f"  {name:28s} {nodes:5d} {links:5d} {degree:6.2f} {diameter:8d}"
    #
    # Write your code here, then delete the next line.
    raise NotCompletedError("TODO 7: size of some real networks")

In [ ]:
run(exercise_7)                # runs exercise 7 and prints its results

## When you have finished

1. **Compare** the results of exercise 2 with the fibre cuts of handout 3, section 5, where every link has 60 wavelengths. Does a cut that leaves the network connected always allow a routing when the capacity is limited?
2. **Save** your notebook: *File > Save a copy in Drive*, or *File > Download > Download .ipynb* in Colab.
3. **The script version.** The same functions are in `labs/lab1_exercises.py`, which runs with the real PyNet2Plan from a terminal: `python labs/lab1_exercises.py` runs all the exercises, and `python labs/lab1_exercises.py --only 2` only one. The functions of the toolbox have the same names as those of PyNet2Plan, so the body of your functions can be copied from the notebook into the script.
4. **Next.** Handout 3 solves the splittable flow formulation on the same network and traffic: an ILP, written with PuLP, that reads the graph with the calls of this notebook.